# ASL Data Exploration with a Kaggle Remote Kernel
Select the Python kernel from your connected Kaggle Jupyter Server in VS Code, then run the cells in order.
This notebook is self-contained: no remote copy of asl.py, local config.json, or local .venv is required.
This notebook inspects data and creates plots without training a model. Outputs are saved to /kaggle/working/asl-project.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INPUT_DIR = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/asl-project')
print('Python:', sys.executable)
if not INPUT_DIR.is_dir():
    raise RuntimeError('Select your Kaggle remote kernel; /kaggle/input is not available in this runtime.')
print('Input folders:', [p.name for p in INPUT_DIR.iterdir()])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)


In [ ]:
# To specify the dataset manually, use Path('/kaggle/input/competitions/asl-signs').
# Search only a few directory levels to avoid traversing every parquet file.
DATA_DIR = None
if DATA_DIR is None:
    candidates = []
    for pattern in ('train.csv', '*/train.csv', '*/*/train.csv', '*/*/*/train.csv'):
        for csv_path in INPUT_DIR.glob(pattern):
            if (csv_path.parent / 'sign_to_prediction_index_map.json').is_file():
                candidates.append(csv_path.parent)
    candidates = sorted(set(candidates))
    if not candidates:
        raise FileNotFoundError(
            'ASL competition data was not found. In your Kaggle notebook, use Add Input to add '
            'Google - Isolated Sign Language Recognition (asl-signs), then rerun this cell.'
        )
    if len(candidates) > 1:
        raise RuntimeError(f'Multiple dataset directories found; set DATA_DIR explicitly above: {candidates}')
    DATA_DIR = candidates[0]
print('Data directory:', DATA_DIR)

table = pd.read_csv(DATA_DIR / 'train.csv')
labels = json.loads((DATA_DIR / 'sign_to_prediction_index_map.json').read_text())
required = {'path', 'sign', 'participant_id'}
if not required.issubset(table.columns):
    raise ValueError(f'train.csv is missing required columns: {required - set(table.columns)}')
if table.empty:
    raise ValueError('train.csv contains no samples')
if sorted(labels.values()) != list(range(len(labels))):
    raise ValueError('Label indices must be contiguous and start at zero')
if not table['sign'].isin(labels).all():
    raise ValueError('train.csv contains labels missing from the official label mapping')
print('Samples:', len(table))
print('Classes:', len(labels))
print('Signers:', table['participant_id'].nunique())
display(table.head())
display(table.groupby('participant_id').size().rename('samples').to_frame())


In [ ]:
def load_frames(path):
    import numpy as np
    import pandas as pd
    data = pd.read_parquet(path, columns=["x", "y", "z"]).to_numpy(dtype=np.float32)
    if len(data) == 0 or len(data) % 543:
        raise ValueError(f"Invalid landmark rows in {path}: {len(data)}")
    return data.reshape(-1, 543, 3)

def preprocess(frames, length):
    """Official row order: face, left hand, pose, right hand.

    Use xy from both hands. Retain a validity mask; normalize each frame
    relative to the visible hand centroid with a shared xy scale.
    """
    import numpy as np
    hands = frames[:, list(range(468, 489)) + list(range(522, 543)), :2]
    valid = np.isfinite(hands).all(axis=-1)
    useful = valid.any(axis=1)
    hands, valid = hands[useful], valid[useful]
    if not len(hands):
        return np.zeros((length, 126), dtype=np.float32)
    safe = np.where(valid[..., None], hands, 0)
    count = np.maximum(valid.sum(axis=1), 1)[:, None, None]
    center = safe.sum(axis=1, keepdims=True) / count
    centered = np.where(valid[..., None], safe - center, 0)
    scale = np.sqrt((centered ** 2).sum(axis=(1, 2), keepdims=True) / (count * 2))
    normalized = centered / np.maximum(scale, 1e-6)
    features = np.concatenate([normalized.reshape(len(hands), 84), valid], axis=1)
    # Uniform resampling also repeats frames for short clips.
    indices = np.rint(np.linspace(0, len(features) - 1, length)).astype(int)
    return features[indices].astype(np.float32)


In [ ]:
row = table.iloc[0]
sample_path = DATA_DIR / row['path']
frames = load_frames(sample_path)
features = preprocess(frames, length=32)
print('Sample:', sample_path)
print('Sign:', row['sign'], '| class index:', labels[row['sign']])
print('Raw shape:', frames.shape)
print('Missing fraction:', float(np.isnan(frames).mean()))
print('Processed shape:', features.shape)
assert features.shape == (32, 126)
assert np.isfinite(features).all()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, index in [('left wrist', 468), ('right wrist', 522)]:
    ax.plot(frames[:, index, 0], label=name)
ax.set(xlabel='Frame', ylabel='Wrist x coordinate', title=str(row['sign']))
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'first_sample_wrist_motion.png', dpi=150)
plt.show()

# Display the frame with the most visible hand landmarks.
left = frames[:, 468:489, :2]
right = frames[:, 522:543, :2]
visible = np.isfinite(np.concatenate([left, right], axis=1)).all(axis=-1).sum(axis=1)
frame_index = int(visible.argmax())
fig, ax = plt.subplots(figsize=(6, 5))
for name, hand in [('left hand', left), ('right hand', right)]:
    ax.scatter(hand[frame_index, :, 0], hand[frame_index, :, 1], label=name)
ax.invert_yaxis()
ax.set_aspect('equal', adjustable='datalim')
ax.set(title=f"{row['sign']} | frame {frame_index}", xlabel='x', ylabel='y')
ax.legend()
fig.tight_layout()
plt.show()


## Next Steps
Confirm that the sample details, class count, and hand plots appear above. Next, split training and validation data by participant_id and train a small CNN.
This exploration notebook does not require a GPU. Files under /kaggle/working depend on the Kaggle session lifecycle; download important artifacts or save a version with outputs before ending the session.
